In [3]:
import csv
import json
import pandas as pd
import numpy as np
import requests
import time
!mkdir data

## Task 1.1 — Read the file the foundations way

In [4]:
accepted = []
rejected = []

def parse_row(row):
  parsed = row.copy()
  parsed['age'] = int(row['age'])
  parsed['fee_inr'] = int(row['fee_inr'])
  parsed['prior_no_shows'] = int(row['prior_no_shows'])
  parsed['distance_km'] = float(row['distance_km'])
  return parsed

with open('data/appointments.csv', 'r', newline='', encoding ='utf-8') as f:
  reader = csv.DictReader(f)
  header = reader.fieldnames
  print(f"Header: {header}")

  # row count
  rows = list(reader)
  print(f"Toatal row count: {len(rows)}")

  for row in rows:
    try:
      parsed_row = parse_row(row)
      accepted.append(parsed_row)
    except ValueError:
      rejected.append(row)

print(f"Accepted rows: {len(accepted)}, Rejected rows: {len(rejected)}")

# save in json
with open('rejected_rows.json', 'w', encoding='utf-8') as f:
  json.dump(rejected, f, indent=2)

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Toatal row count: 1020
Accepted rows: 988, Rejected rows: 32


## Task 1.2 — Inspect and clean, pass 1

In [5]:
df = pd.read_csv('data/appointments.csv')

print(f"Shape: {df.shape}")
print("\nInfo:")
df.info()
print("\nNulls:\n", df.isnull().sum())
print("\nDuplicates: ", df.duplicated().sum())

#drop duplicate rows
df = df.drop_duplicates()
print(f"\nRow count after dropping duplicates: {len(df)}")

#clean booking_channel
df['booking_channel'] = df['booking_channel'].str.strip().str.title()

#convert dates
df['appt_date'] = pd.to_datetime(df['appt_date'], errors='coerce')
df['booked_on'] = pd.to_datetime(df['booked_on'])

#dropping rows where appt_date or status is missing
df = df.dropna(subset=['appt_date', 'status'])

print(f"row count after dropping missing appt_date and status: {len(df)}")


Shape: (1020, 14)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Nulls:
 appointment_id      0
patient_id          0
clinic_id     

## Task 1.3 — Clean, pass 2: validity, gaps and outliers

In [6]:
#Treat any age below 1 or above 100 as invalid and set it to missing.
df.loc[(df['age']< 1) | (df['age'] > 100), 'age'] = np.nan

#filling missing age and distance_km with median
age_median = df['age'].median()
df['age'] = df['age'].fillna(age_median).round().astype(int)

dist_median = df['distance_km'].median()
df['distance_km'] = df['distance_km'].fillna(dist_median)

#Compute Q1, Q3, IQR and cap values (np.clip)
def cap_outliers(series):
  Q1 = series.quantile(0.25)
  Q3 = series.quantile(0.75)
  IQR = Q3 -Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR

  #count how many will be capped
  capped_count = ((series < lower_bound) | (series > upper_bound)).sum()

  return np.clip(series, lower_bound, upper_bound), capped_count

df['distance_km'], dist_capped = cap_outliers(df['distance_km'])
df['fee_inr'], fee_capped  = cap_outliers(df['fee_inr'])

#Rounding
df['distance_km'] = df['distance_km'].round(1)
df['fee_inr'] = df['fee_inr'].round().astype(int)

print(f"Values_capped - distance_km: {dist_capped}, fee_inr: {fee_capped}")


Values_capped - distance_km: 26, fee_inr: 3


#Task 1.4 — Features and a NumPy summary

In [7]:
#create new features
df['lead_days'] = (df['appt_date']- df['booked_on']).dt.days
df['no_show'] = (df['status'] == 'No-Show').astype(int)

#Extract numpy array for specific columns
X = df[['age', 'distance_km', 'fee_inr', 'lead_days']].to_numpy()

#mean of each column
means = np.mean(X, axis=0)
print(f"column Means [age, distance_km, fee_inr, lead_days]: {means.round(2)}")

#Rows with lead days > 14
lead_mask = X[:,3] > 14
print(f"rows with lead days > 14: Count = {lead_mask.sum()}, Share = {lead_mask.mean():.3f}")

#z-scores of distance_km (Index 1) and count of |z| > 2
dist_col = X[:, 1]
z_scores = (dist_col - np.mean(dist_col)) / np.std(dist_col)
z_count = (np.abs(z_scores) > 2).sum()
print(f"Rows with |z| > 2 for distance_km: {z_count}")


column Means [age, distance_km, fee_inr, lead_days]: [ 48.17   9.5  635.71  11.82]
rows with lead days > 14: Count = 296, Share = 0.299
Rows with |z| > 2 for distance_km: 53


##Task 1.5 — Enrich the clinics from a live API

In [9]:
clinics = pd.read_csv('data/clinics.csv')
temp_c_list = []

for index,row in clinics.iterrows():
  lat, lon = row['latitude'], row['longitude']
  url = "https://api.open-meteo.com/v1/forecast"
  params = {"latitude":lat, "longitude":lon, "hourly":"temperature_2m", "forecast_days":1}

  temp_c = None
  for attempt in range(2):
    try:
      resp = requests.get(url, params=params, timeout=10)
      if resp.status_code == 200:
        data = resp.json()
        if "hourly" in data and "temperature_2m" in data["hourly"]:
          temp_c = data["hourly"]["temperature_2m"][0]
          break

    except requests.exceptions.RequestException:
      pass
    if attempt == 0:
      time.sleep(1)
  if temp_c is None:
    print(f"Clinic {row['name']} failed to fetch temperature.")
  temp_c_list.append(temp_c)

clinics['temp_c'] = temp_c_list
clinics.to_csv('clinics_enriched.csv', index=False)
print("clinics_enriched.csv saved successfully.")


clinics_enriched.csv saved successfully.


##Task 1.6 — Export the clean table and write the SQL load script

In [10]:
df = df.sort_values('appointment_id')
if pd.api.types.is_datetime64_any_dtype(df['appt_date']):
  df['appt_date'] = df['appt_date'].dt.strftime('%Y-%m-%d')

if pd.api.types.is_datetime64_any_dtype(df['appt_date']):
  df['booked_on'] = df['booked_on'].dt.strftime('%Y-%m-%d')

#arranging columns
cols=[
    'appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date',
    'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel',
    'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show'
]
df = df[cols]

df.to_csv('appointments_clean.csv', index=False)

print(f"Unique appointment_ids: {df['appointment_id'].nunique()}")
print("Missing values in final dataframe:\n", df.isnull().sum().sum())

#SQL script generation
clinics_df = pd.read_csv('clinics_enriched.csv')

def format_sql(val):
  if pd.isna(val):
    return 'NULL'
  elif isinstance(val, str):
    return f"'{val}'"
  else:
    return str(val)

with open('load_data.sql','w', encoding='utf-8') as f:
  for _,row in clinics_df.iterrows():
    vals = [format_sql(x) for x in row]
    f.write(f"Insert into clinics values ({','.join(vals)});\n")

  for _,row in df.iterrows():
    vals = [format_sql(x) for x in row]
    f.write(f"Insert into appointments values ({','.join(vals)});\n")
total_inserts = len(clinics_df) + len(df)

print(f"load_data.sql generated with {total_inserts} Insert statements.")



Unique appointment_ids: 990
Missing values in final dataframe:
 0
load_data.sql generated with 998 Insert statements.
